01 — Data Inspection

**Purpose:** Load the Telco Customer Churn dataset and understand
its structure before any cleaning or modeling.

**Dataset:** data/raw/telco_customer_churn.csv

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

In [ ]:
DATA_PATH = "../data/raw/telco_customer_churn.csv"

df = pd.read_csv(DATA_PATH)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0]

if len(missing) == 0:
    print("No missing values detected by pandas.")
else:
    print("Missing values per column:")
    print(missing)

In [ ]:
blank_total_charges = (df["TotalCharges"].astype(str).str.strip() == "").sum()

print("Blank TotalCharges entries:", blank_total_charges)
print("TotalCharges dtype:", df["TotalCharges"].dtype)

In [ ]:
churn_counts = df["Churn"].value_counts()
churn_pct = df["Churn"].value_counts(normalize=True) * 100

print("Churn counts:")
print(churn_counts)

print()
print("Churn percentage:")
print(churn_pct.round(2))

In [ ]:
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical columns:", len(categorical_cols))
print()

for col in categorical_cols:
    unique_vals = df[col].nunique()
    print(f"{col:25s} → {unique_vals} unique values")

In [ ]:
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns

print("Numeric columns:", list(numeric_cols))
print()

for col in numeric_cols:
    print(
        f"{col:20s} "
        f"min={df[col].min()}   "
        f"max={df[col].max()}   "
        f"mean={df[col].mean():.2f}"
    )

## Day 2 — Findings Summary

**Dataset shape:** 7,043 rows × 21 columns.

**Target column:** `Churn` (Yes / No), currently stored as text.

**Class distribution:** Imbalanced — approximately 26–27% of
customers churned. This must be considered when choosing
evaluation metrics later.

**Data quality issues found so far:**

1. `TotalCharges` is stored as text (`object`) instead of a number.
2. `TotalCharges` contains a small number of blank entries that
   pandas does not detect as missing values.
3. `customerID` is a unique identifier with no predictive value
   and should be removed before modeling.
4. `Churn` needs to be converted from Yes/No text to 1/0 for
   machine learning.

**Next step (Day 3):** Clean the dataset — fix data types, handle
blanks, drop the ID column, and save a cleaned copy to
`data/processed/`.

---
## Day 3 — Data Cleaning
---

In [ ]:
import pandas as pd
import numpy as np

# Load the raw dataset fresh for cleaning
df = pd.read_csv("../data/raw/telco_customer_churn.csv")

print("Shape before cleaning:", df.shape)
df.head(3)

In [ ]:
# Check all column types before cleaning
df.dtypes

In [ ]:
# Find rows where TotalCharges is blank
blank_mask = df["TotalCharges"].astype(str).str.strip() == ""

print("Number of blank TotalCharges rows:", blank_mask.sum())
print()
print("The rows with blank TotalCharges:")
df[blank_mask][["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

In [ ]:
# Step 1: Replace blank spaces with NaN
df["TotalCharges"] = df["TotalCharges"].str.strip()
df["TotalCharges"] = df["TotalCharges"].replace("", np.nan)

# Step 2: Convert to numeric
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

# Step 3: Fill the 11 blank values with 0.0
df["TotalCharges"] = df["TotalCharges"].fillna(0.0)

# Verify
print("TotalCharges dtype after fix:", df["TotalCharges"].dtype)
print("Remaining NaN in TotalCharges:", df["TotalCharges"].isnull().sum())

In [ ]:
print("Before conversion:")
print(df["Churn"].value_counts())
print("Dtype:", df["Churn"].dtype)

In [ ]:
# Convert Churn: Yes → 1, No → 0
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# Verify
print("After conversion:")
print(df["Churn"].value_counts())
print("Dtype:", df["Churn"].dtype)

In [ ]:
# Drop customerID - it is an identifier, not a feature
print("Columns before drop:", df.shape[1])

df = df.drop(columns=["customerID"])

print("Columns after drop:", df.shape[1])
print("customerID still in df:", "customerID" in df.columns)

In [ ]:
# Check for duplicate rows
duplicates = df.duplicated().sum()

print("Number of duplicate rows:", duplicates)

In [ ]:
# Inspect the duplicate rows
duplicate_rows = df[df.duplicated(keep=False)]

print("Total duplicate rows:", len(duplicate_rows))
duplicate_rows.head(30)

In [ ]:
# Check how many duplicate groups there are
print("Duplicate rows excluding first occurrence:", df.duplicated().sum())
print("Number of completely duplicated records:", df.duplicated(keep=False).sum())

In [ ]:
# Remove duplicate rows, keeping the first occurrence
before = df.shape[0]

df = df.drop_duplicates()

after = df.shape[0]

print("Rows before removing duplicates:", before)
print("Rows after removing duplicates:", after)
print("Rows removed:", before - after)

In [ ]:
print("Remaining duplicate rows:", df.duplicated().sum())

In [ ]:
# Final missing value check after cleaning
missing = df.isnull().sum()

print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "No missing values found.")

In [ ]:
# Final shape check
print("Final cleaned shape:", df.shape)
print()

print("Column data types:")
print(df.dtypes)

In [ ]:
# Preview the clean dataset
df.head(5)

In [ ]:
# Check basic statistics for numeric columns
df.describe()

In [ ]:
import os

# Make sure the processed folder exists
os.makedirs("../data/processed", exist_ok=True)

# Save the cleaned dataset
output_path = "../data/processed/cleaned_churn_data.csv"
df.to_csv(output_path, index=False)

print("Cleaned dataset saved to:", output_path)

# Verify it was saved correctly by reloading
df_check = pd.read_csv(output_path)

print("Rows:", df_check.shape[0])
print("Columns:", df_check.shape[1])
print("TotalCharges dtype in saved file:", df_check["TotalCharges"].dtype)
print("Churn dtype in saved file:", df_check["Churn"].dtype)